# Preparar corpus de la Pokedex

Ejecuta las celdas de arriba a abajo. El cuaderno genera o reutiliza las fichas y crea el indice de ChromaDB para la aplicacion.

In [ ]:
import json
import time
from pathlib import Path

import chromadb
import ollama
import pandas as pd
import requests

CARPETA = Path.cwd()
FICHERO_FICHAS = CARPETA / "fichas_pokemon.json"
MODELO_EMBEDDINGS = "embeddinggemma"

In [ ]:
def descargar_descripcion(id_pokemon, sesion):
    """Devuelve la descripcion en castellano desde la PokeAPI."""
    url = f"https://pokeapi.co/api/v2/pokemon-species/{id_pokemon}/"
    try:
        respuesta = sesion.get(url, timeout=10)
        respuesta.raise_for_status()
        datos = respuesta.json()
    except requests.RequestException:
        return ""

    for entrada in datos["flavor_text_entries"]:
        if entrada["language"]["name"] == "es":
            return " ".join(entrada["flavor_text"].split())
    return ""

In [ ]:
def construir_fichas():
    """Crea una ficha de texto por Pokemon con datos y descripcion."""
    df = pd.read_csv(CARPETA / "pokemon.csv")
    sesion = requests.Session()
    fichas = []

    for _, p in df.iterrows():
        tipos = p["type_1"] if pd.isna(p["type_2"]) else f"{p['type_1']} y {p['type_2']}"
        ficha = (
            f"{p['name']} es el pokemon numero {int(p['id'])}. "
            f"Es de tipo {tipos}. "
            f"Estadisticas de {p['name']}: {int(p['hp'])} puntos de salud (HP), "
            f"{int(p['attack'])} de ataque, {int(p['defense'])} de defensa, "
            f"{int(p['special_attack'])} de ataque especial, "
            f"{int(p['special_defense'])} de defensa especial y "
            f"{int(p['speed'])} de velocidad. Total de estadisticas: {int(p['total'])}. "
            f"Mide {p['height_m']} metros y pesa {p['weight_kg']} kilos. "
            f"{'Es un pokemon legendario. ' if p['legendary'] else ''}"
        )
        descripcion = descargar_descripcion(int(p["id"]), sesion)
        if descripcion:
            ficha += f"Descripcion de la Pokedex: {descripcion}"
        fichas.append({"id": int(p["id"]), "nombre": p["name"], "texto": ficha})
        print(f"  {int(p['id']):>3} {p['name']:<12} {'ok' if descripcion else 'SIN descripcion'}")

    return fichas

In [ ]:
def indexar(fichas):
    """Vectoriza las fichas y las guarda en ChromaDB."""
    cliente = chromadb.PersistentClient(path=str(CARPETA / "chroma_pokedex"))
    coleccion = cliente.get_or_create_collection(
        name="fichas_pokemon",
        metadata={"hnsw:space": "cosine"},
    )

    for inicio in range(0, len(fichas), 32):
        lote = fichas[inicio : inicio + 32]
        textos = [f"title: none | text: {f['texto']}" for f in lote]
        vectores = ollama.embed(model=MODELO_EMBEDDINGS, input=textos)["embeddings"]
        coleccion.upsert(
            ids=[f"pokemon_{f['id']}" for f in lote],
            embeddings=vectores,
            documents=[f["texto"] for f in lote],
            metadatas=[{"nombre": f["nombre"]} for f in lote],
        )
        print(f"  indexadas {min(inicio + 32, len(fichas))}/{len(fichas)}")

    return coleccion.count()

In [ ]:
if FICHERO_FICHAS.exists():
    print(f"Ya existe {FICHERO_FICHAS.name}: uso las fichas guardadas.")
    fichas = json.loads(FICHERO_FICHAS.read_text(encoding="utf-8"))
else:
    print("Descargando descripciones de la PokeAPI...")
    inicio = time.time()
    fichas = construir_fichas()
    FICHERO_FICHAS.write_text(
        json.dumps(fichas, ensure_ascii=False, indent=2), encoding="utf-8"
    )
    print(f"Fichas guardadas en {FICHERO_FICHAS.name} ({time.time() - inicio:.0f} s)")

print("Indexando en ChromaDB...")
total = indexar(fichas)
print(f"Listo: {total} fichas indexadas en chroma_pokedex/.")